# Customer Segmentation using K-Means Clustering


**Dataset:** Mall Customers - Kaggle

**Problem Statement:** to understand the customers like who can be target customer so that the sense can be given to marketing team and plan the strategy accordingly.

**Tools:** Python -- Pandas -- Scikit-learn -- Plotly

# 1. Setup + Data Loading

In [46]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings("ignore")

# Load the dataset
df = pd.read_csv('data/Mall_Customers.csv')    
print("Dataset loaded")
print(f'Shape:{df.shape[0]} rows  × {df.shape[1]} columns')

Dataset loaded
Shape:200 rows  × 5 columns


In [42]:
# First 10 rows of the dataset
print(df.head(10))

   CustomerID  Gender  Age  Annual Income (k$)  Spending Score (1-100)
0           1    Male   19                  15                      39
1           2    Male   21                  15                      81
2           3  Female   20                  16                       6
3           4  Female   23                  16                      77
4           5  Female   31                  17                      40
5           6  Female   22                  17                      76
6           7  Female   35                  18                       6
7           8  Female   23                  18                      94
8           9    Male   64                  19                       3
9          10  Female   30                  19                      72


In [43]:
# Data type, null, duplicates
print('Data Info')
df.info()
print('\nNull Values')
print(df.isnull().sum())
print('\nDuplicate Rows')
print(f'Duplicates: {df.duplicated().sum()}')

Data Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   CustomerID              200 non-null    int64 
 1   Gender                  200 non-null    object
 2   Age                     200 non-null    int64 
 3   Annual Income (k$)      200 non-null    int64 
 4   Spending Score (1-100)  200 non-null    int64 
dtypes: int64(4), object(1)
memory usage: 7.9+ KB

Null Values
CustomerID                0
Gender                    0
Age                       0
Annual Income (k$)        0
Spending Score (1-100)    0
dtype: int64

Duplicate Rows
Duplicates: 0


### Observation result

- The data provide **200 customers** and **5 columns** - no duplicates or null values [clean dataset]
- `CustomerID` is indentifier [will be dropped]

- `Annual Income` ranges 15-137k
- `Spending Score` ranges 1-99 (**scaling is necessary**)

 - `Gender` is categorical (Male/Female) -> used in EDA, not in clustering

# 2. Exploratory Data Analysis (EDA)
**purpose: explore the data visually**
1. distributions of each variables
2. spot patterns, outliers, or skewness
3. the amount of natural groups might exist

In [48]:
# Renaming the columns for better readability
df.rename(columns={
    'Annual Income (k$)': 'Income',
    'Spending Score (1-100)': 'SpendingScore',
    'Genre': 'Gender'
}, inplace=True)

print('Columns:', df.columns.tolist())

Columns: ['CustomerID', 'Gender', 'Age', 'Income', 'SpendingScore']


In [49]:
# Distribution of numeric features
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
)

for i, col in enumerate(['Age', 'Income', 'SpendingScore'], 1):
    fig.add_trace(
        go.Histogram(x=df[col], name=col, marker_color=["#1723D7",'#EF553B','#00CC96'][i-1], nbinsx=20),
        row=1, col=i
    )

fig.update_layout(
    title_text='Distribution of Numeric Features',
    showlegend=False,
    height=400,
    template='plotly_white'
)
fig.show()

In [50]:
# Gender distribution
gender_counts = df['Gender'].value_counts().reset_index()
gender_counts.columns = ['Gender', 'Count']

fig = px.pie(
    gender_counts, values='Count', names='Gender',
    title='Gender Distribution',
    color_discrete_sequence=['#636EFA', '#EF553B'],
    template='plotly_white'
)
fig.show()

In [ ]:

# Key scatter: Income vs Spending Score [main clustering target]
fig =px.scatter(
    df,
    x='Income',
    y='SpendingScore',
    color='Gender',
    hover_data=['Age', 'CustomerID'],
    title='Annual Income vs Spending Score (colored by Gender)',
    labels={'Income': 'Annual Income (k$)', 'SpendingScore': 'Spending Score (1-100)'},
    template='plotly_white',
    color_discrete_sequence=['#636EFA', '#EF553B']
)
fig.update_traces(marker=dict(size=8, opacity=0.7))
fig.show()

In [51]:
#Boxplot to check for outliers across numeric features
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Age Outliers', 'Income Outliers', 'Spending Score Outliers')
)

for i, col in enumerate(['Age', 'Income', 'SpendingScore'], 1):
    fig.add_trace(
        go.Box(y=df[col], name=col, marker_color=["#1723D7",'#EF553B','#00CC96'][i-1]),
        row=1, col=i
    )
fig.update_layout(
    title_text='Boxplots for Outlier Checking',
    showlegend=False,
    height=400,
    template='plotly_white'
)
fig.show()

In [52]:
# Correlation heatmap
corr = df[['Age', 'Income', 'SpendingScore']].corr().round(2)

fig = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    title='Correlation Heatmap',
    labels={'color': 'Correlation Coefficient'},
    zmin=-1, zmax=1,
    template='plotly_white'
)
fig.show()

### Observation Result

- `Income` vs `SpendingScore` scatterplot show **5 groups forming**
- No significant outliers detected from boxplots
- Low correlation `Income`  and `SpendingScore` (-0.01) **[independent variables]** --. good for clustering
- Gender is balanced (56% Female, 44% Male)


# 3. Preprocessing + Feature Scaling

K-Means uses **Euclidean distance* to assign points to clusters
Since `Income`: 15–137 vs `Score`: 1–99 **{different scales}**, the larger scale will  **dominate** the calcualtion unfairly. So normalize then using `StandardScaler` (mean=0, std=1)


In [55]:
# Select features for clustering
features = ['Income', 'SpendingScore']
X = df[features].copy()

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Quick check
print('Before scaling:')
print(X.describe().round(2))
print('\nAfter scaling (mean ≈ 0, std ≈ 1):')
print(pd.DataFrame(X_scaled, columns=features).describe().round(2))

Before scaling:
       Income  SpendingScore
count  200.00         200.00
mean    60.56          50.20
std     26.26          25.82
min     15.00           1.00
25%     41.50          34.75
50%     61.50          50.00
75%     78.00          73.00
max    137.00          99.00

After scaling (mean ≈ 0, std ≈ 1):
       Income  SpendingScore
count  200.00         200.00
mean    -0.00          -0.00
std      1.00           1.00
min     -1.74          -1.91
25%     -0.73          -0.60
50%      0.04          -0.01
75%      0.67           0.89
max      2.92           1.89


### Observaation result

- After scaling, both features have **mean** = 0 and **std** = 1. So both features contibute equally to the **distance calculation**

**Conclusion:** `Income` and `SpendingScore` will be used for 2D clustering

# 4. Finding the Optimal K (Elbow Method)

**K* is the number of clusters that will be required for K-Means
**Elbow Method** helps us to find the best **K** by **plotting inertia** (sum of squared distances from each points to its cluster center) for **K** = 1-10

*elbow* -- the optimal K -- is where the curve bends sharply

In [57]:
# Elbow Method
inertia = []
K_range = range(1, 20)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

# Plot elbow curve
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(K_range),
    y=inertia,
    mode='lines+markers',
    marker=dict(size=8, color='#636EFA'),
    line=dict(width=2, color='#636EFA'),
    name='Inertia'
))

# Highlight elbow at K=5
fig.add_trace(go.Scatter(
    x=[5], y=[inertia[4]],
    mode='markers',
    marker=dict(size=14, color='#EF553B', symbol='star'),
    name='Optimal K = 5'
))

fig.update_layout(
    title='Elbow Method — Finding Optimal K',
    xaxis_title='Number of Clusters (K)',
    yaxis_title='Inertia (Within-cluster Sum of Squares)',
    template='plotly_white',
    height=450
)
fig.show()

print(f'Inertia values: {[round(i, 1) for i in inertia]}')

Inertia values: [400.0, 269.7, 157.7, 108.9, 65.6, 55.1, 44.9, 37.2, 32.4, 30.0, 25.9, 23.9, 21.2, 18.9, 17.3, 16.1, 14.9, 14.0, 13.1]


### Observation result

- The curve drops steeply from **K** = 1 to **K** = 5, then it flattens out
- The **elbow is at K=5** -> matches the visual intuition from EDA scatter plot

**Conclusion:** use **K = 5**

# 5. Running K-Means + Assigning Clusters 
Adding cluster labels to te original dataframe and assign each customer to a cluster with **K=5**

In [59]:
# Fit K-Means with K = 5
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Cluster centers (in original scale)
centers_scaled = kmeans.cluster_centers_
centers_original = scaler.inverse_transform(centers_scaled)
centers_df = pd.DataFrame(centers_original, columns=features)
centers_df.index.name = 'Cluster'
centers_df = centers_df.round(1)

print('Cluster Centers (original scale):')
print(centers_df)
print('\nCustomers per cluster:')
print(df['Cluster'].value_counts().sort_index())

Cluster Centers (original scale):
         Income  SpendingScore
Cluster                       
0          55.3           49.5
1          86.5           82.1
2          25.7           79.4
3          88.2           17.1
4          26.3           20.9

Customers per cluster:
Cluster
0    81
1    39
2    22
3    35
4    23
Name: count, dtype: int64


### Observation result

- Each of the 200 customers now has a cluster label (0-4)
- Cluster center table show the **average income and spending score** for each group

# 6. Visualizing Clusters

Each dot is a customer, colored by their assigned cluster 

*Cluster centers are shows as stars*

In [64]:
# Assign segment names (will be finalized in Step 7)
cluster_names = {
    0: 'Cluster 0',
    1: 'Cluster 1',
    2: 'Cluster 2',
    3: 'Cluster 3',
    4: 'Cluster 4'
}
df['Segment'] = df['Cluster'].map(cluster_names)

# Scatter plot
fig = px.scatter(
    df,
    x='Income',
    y='SpendingScore',
    color='Segment',
    hover_data=['Age', 'Gender', 'CustomerID'],
    title='Customer Segments — K-Means Clustering (K=5)',
    labels={'Income': 'Annual Income (k$)', 'SpendingScore': 'Spending Score (1-100)'},
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_traces(marker=dict(size=9, opacity=0.8))

# Add cluster centers as stars
fig.add_trace(go.Scatter(
    x=centers_df['Income'],
    y=centers_df['SpendingScore'],
    mode='markers',
    marker=dict(symbol='star', size=18, color='silver', line=dict(width=1, color='white')),
    name='Cluster Centers'
))

fig.update_layout(height=550)
fig.show()

In [65]:
# Cluster profile summary
profile = df.groupby('Cluster')[['Age', 'Income', 'SpendingScore']].mean().round(1)
profile['Count'] = df.groupby('Cluster')['CustomerID'].count()
print('Cluster Profile Summary:')
profile

Cluster Profile Summary:


,Age,Income,SpendingScore,Count
Cluster,,,,
0,42.7,55.3,49.5,81
1,32.7,86.5,82.1,39
2,25.3,25.7,79.4,22
3,41.1,88.2,17.1,35
4,45.2,26.3,20.9,23


In [66]:
# Bar chart: avg income and spending per cluster
fig = make_subplots(rows=1, cols=2, subplot_titles=['Avg Annual Income by Cluster', 'Avg Spending Score by Cluster'])

colors = px.colors.qualitative.Bold[:5]

fig.add_trace(
    go.Bar(x=profile.index.astype(str), y=profile['Income'], marker_color=colors, name='Income'),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=profile.index.astype(str), y=profile['SpendingScore'], marker_color=colors, name='SpendingScore'),
    row=1, col=2
)

fig.update_layout(
    title_text='Cluster Profiles — Income & Spending Score',
    showlegend=False,
    height=400,
    template='plotly_white'
)
fig.show()

# Business Interpretation

**Actionable customer personas**.

Based on the cluster centers (Income vs Spending Score):

| Cluster | Income | Spending Score | Segment Name | Business Action |
|---------|--------|----------------|--------------|------------------|
| 0 | Medium | Medium | 🟡 Standard Customers | Maintain loyalty programs |
| 1 | High | High | 🟢 Prime Targets | VIP rewards, premium products |
| 2 | Low | High | 🔵 Impulsive Buyers | Flash sales, limited-time offers |
| 3 | High | Low | 🟠 Careful Spenders | Emphasize value, discounts, ROI |
| 4 | Low | Low | 🔴 Budget Conscious | Budget-friendly promotions |


In [69]:
# Map actual cluster numbers to segment names based on your cluster centers
# Run the profile summary above first, then adjust this mapping accordingly

# Example mapping (adjust cluster numbers based on your results):
segment_map = {
    0: '🟡 Standard Customers',
    1: '🟢 Prime Targets',
    2: '🔵 Impulsive Buyers',
    3: '🟠 Careful Spenders',
    4: '🔴 Budget Conscious'
}

df['Segment'] = df['Cluster'].map(segment_map)

# Final visualization with named segments
fig = px.scatter(
    df,
    x='Income',
    y='SpendingScore',
    color='Segment',
    hover_data=['Age', 'Gender'],
    title='Customer Segmentation — Final Named Segments',
    labels={'Income': 'Annual Income (k$)', 'SpendingScore': 'Spending Score (1-100)'},
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_traces(marker=dict(size=9, opacity=0.8))
fig.add_trace(go.Scatter(
    x=centers_df['Income'],
    y=centers_df['SpendingScore'],
    mode='markers',
    marker=dict(symbol='star', size=18, color='black', line=dict(width=1, color='white')),
    name='Cluster Centers'
))
fig.update_layout(height=550)
fig.show()

In [70]:
# Final segment summary table
final_summary = df.groupby('Segment').agg(
    Count=('CustomerID', 'count'),
    Avg_Age=('Age', 'mean'),
    Avg_Income=('Income', 'mean'),
    Avg_SpendingScore=('SpendingScore', 'mean')
).round(1)

print('=== Final Customer Segment Summary ===')
final_summary

=== Final Customer Segment Summary ===


,Count,Avg_Age,Avg_Income,Avg_SpendingScore
Segment,,,,
🔴 Budget Conscious,23,45.2,26.3,20.9
🔵 Impulsive Buyers,22,25.3,25.7,79.4
🟠 Careful Spenders,35,41.1,88.2,17.1
🟡 Standard Customers,81,42.7,55.3,49.5
🟢 Prime Targets,39,32.7,86.5,82.1


## Conclusion

Prime targets is the most attractive target segment because they have both **high income** and **high spending behavior**. They actively shop at the mall and contribute significantly to overall revenue. In addition, they have strong potential to become loyal customers, making them a valuable long-term asset for the business. This segment is suitable for **VIP memberships**, **exclusive promotions**, and **personalized shopping recommendations** to strengthen customer retention and maximize customer lifetime value.

There is also a segment of customers with **high income but low spending behavior**. Although they have strong purchasing power, they currently contribute relatively little to revenue. This group represents the **largest untapped revenue opportunity** for the mall, as even a small increase in their spending could generate substantial returns. Therefore, **premium product campaigns**, **targeted marketing efforts**, and **long-term loyalty incentives** could be effective strategies to encourage higher spending.

On the other hand, **impulsive buyers** segements is notably younger, aged around 25 (youngest among all segements). They are an attractive target market because **flash sales**, **limited-time offers**, and **seasonal promotions** can easily influence their purchasing decisions. As a result, this segment has strong potential to generate short-term sales growth.

Meanwhile, some customers are more conscious of their spending because their income is relatively low. As a result, they are more likely to respond to **discount promotions**, **budget-friendly products**, and **value-for-money offers**. Maintaining affordability is important to retain customers within this segment.

Lastly, there is the **standard customer** segment, which represents customers with average income and spending behavior. While they may not be the highest-value customers individually, they form the largest portion of the customer base and provide a stable source of revenue. This group can be retained and further developed through **customer retention programs**, **cross-selling**, and **upselling strategies**.

Overall, the analysis suggests that customer spending behavior is not solely determined by income level. Customers with similar income levels can exhibit very different purchasing patterns. Therefore, segment-specific marketing strategies are likely to be more effective than a one-size-fits-all approach in maximizing revenue and improving customer satisfaction.